# Robust Optimisation via Sum of Largest Eigenvalues

Two problems:
  1. **Robust Design (SDP)**: Minimise the sum of the top-k eigenvalues of a matrix A subject to convex constraints.

  2. **Worst-case analysis**: Find the worst-case sum of top-$k$ eigenvalues of a covariance matrix under spectral-norm bounded perturbations.

## Required packages

In [ ]:
import random
import numpy as np
import cvxpy as cp
import cvxopt

## Data generation

In [ ]:
random.seed(2026)

n = 6 # matrix dimension
k = 2 # number of top eigenvalues to sum
epsilon = 0.3 # perturbation budget (spectral norm)

# Generate a random PSD base covariance matrix Sigma_0
A_random = np.random.randn(n, n)
sigma_0 = A_random @ A_random.T / n + 0.5 * np.eye(n)   # guaranteed PSD

# Sum of top-k eigenvalues
def sum_top_k_eigs(M, k):
    eigs = np.linalg.eigvalsh(M) # sorted ascending
    return np.sum(eigs[-k:])

## Robust design (SDP)

Design a symmetric PSD matrix $A$ that minimizes the sum of its top-$k$ eigenvalues subject to:

$tr(A) = 1 \quad$ (normalisation — energy constraint)
  
$A  \succeq  0$
  
$\text{diag}(A) \geq 0.02 \quad$ (each diagonal entry bounded below)

### Reformulation:

$\min_{A, t, S} [kt + tr(S)]$

subject to:
  
$A - tI \preceq S$
    
$S  \succeq  0$
    
$tr(A) = 1$
    
$A  \succeq  0$

$\text{diag}(A) \geq 0.02$

### Interpretation

Find the "most spread-out" PSD matrix on the trace-simplex (minimise concentration of variance in any $k$ directions).

In [ ]:
A = cp.Variable((n, n), PSD=True)
S = cp.Variable((n, n), PSD=True)
t = cp.Variable()

objective = cp.Minimize(k*t + cp.trace(S))

constraints = [
    A - t*np.eye(n) << S,
    cp.trace(A) == 1,
    cp.diag(A) >= 0.02,
]

problem = cp.Problem(objective, constraints)
problem.solve()

A_solution = A.value
if A_solution is not None:
    eigs_opt = np.sort(np.linalg.eigvalsh(A_solution))[::-1]
    print(f"Sum of top-{k} eigenvalues: {problem.value}")
    print(f"Eigenvalues of A: {np.round(eigs_opt, 3)}")
    print(f"Trace of A: {np.round(np.trace(A_solution))}")
    print(f"")
    print(f"A matrix:")
    print(np.round(A_solution, 3))
else:
    print("Solver did not find a solution.")

## Worst-case analysis

Maximize  sum of top-$k$ eigenvalues of ($\Sigma_0 + \Delta$) subject to:

$||\Delta||_2 \leq \epsilon \quad$(spectral norm)

$\Sigma_0 + \Delta \succeq 0$

### Closed-form worst-case perturbation

For the spectral norm ball, the worst $\Delta$ is:

$\Delta^\ast = \epsilon V_k V_k^T$

where $V_k$ are the top-$k$ eigenvectors of $\Sigma_0$.

### Interpretation

This pushes each of the top-$k$ eigenvalues up by exactly epsilon, and does not affect the remaining eigenvalues.

The worst-case value is therefore:  `sum_k_lambda`$(\Sigma_0) + \epsilon k$

In [ ]:
# Eigen-decomposition of Sigma_0
eigs, V = np.linalg.eigh(sigma_0) # ascending order
eigs_desc = eigs[::-1]
V_desc = V[:, ::-1] # eigenvectors, descending

# Worst-case perturbation: align with top-k eigenvectors
V_k = V_desc[:, :k] # (n x k) top-k eigenvectors
Delta_star = epsilon * V_k @ V_k.T # rank-k, spectral norm = epsilon

Sigma_worst = sigma_0 + Delta_star

nominal_val    = sum_top_k_eigs(sigma_0, k)
worst_case_val = sum_top_k_eigs(Sigma_worst, k)
analytical_val = nominal_val + epsilon * k

print(f"\nNominal sum of top-{k} eigenvalues: {nominal_val:.4f}")
print(f"Worst-case sum of top-{k} eigenvalues: {worst_case_val:.4f}")
print(f"Analytical bound  (nominal + epsilon*k): {analytical_val:.4f}")
print(f"These agree? {np.isclose(worst_case_val, analytical_val, atol=1e-6)}")

print(f"\nWorst-case Delta* (spectral norm = {np.linalg.norm(Delta_star, 2):.4f}, budget = {epsilon}):")
print(np.round(Delta_star, 3))

print(f"\nNominal eigenvalues: {np.round(eigs_desc, 4)}")
print(f"Worst-case eigenvalues: {np.round(np.sort(np.linalg.eigvalsh(Sigma_worst))[::-1], 4)}")
print(f"(Top-{k} each increased by epsilon = {epsilon})")